# 6CS012 - Worksheet 4: Introduction to Keras and Image Classification with FCN
### Building Fully Connected Neural Networks for Devnagari Handwritten Digit Classification
**Prepared By:** Siman Giri {Module Leader - 6CS012}

---
This notebook contains all TensorFlow and Keras code examples from the worksheet.

## Section 2: Introduction to Keras
### Installing Keras and TensorFlow

In [ ]:
# Install Keras and TensorFlow (already pre-installed in Colab)
# !pip install keras tensorflow


In [ ]:
# Identifying the version of Keras
import tensorflow as tf

print(tf.keras.__version__)


3.13.2


---
## Section 2.1: Why Not Just Use NumPy?
### Example 1: Manual Gradient Calculation in NumPy

In [ ]:
import numpy as np


# Simple function f(x) = x^2
def f(x):
    return x**2


# Manual derivative (f'(x) = 2x)
def gradient(x):
    return 2 * x


# Update rule: x = x - learning_rate * gradient
x = 5.0
learning_rate = 0.1
for _ in range(10):  # Manually optimize for 10 steps
    x -= learning_rate * gradient(x)
    print(f"x: {x:.4f}, f(x): {f(x):.4f}")


### Example 2: Gradient Computations with Keras (TensorFlow)

In [ ]:
import tensorflow as tf

x = tf.Variable(5.0)  # Trainable variable
with tf.GradientTape() as tape:
    y = x**2  # y = x^2
grad = tape.gradient(y, x)  # Computes dy/dx automatically
print(grad.numpy())  # Output: 10.0


### Example 3: Matrix Multiplication Speed — NumPy vs TensorFlow on GPU

In [ ]:
import numpy as np
import tensorflow as tf
import time

# Create large random matrices
size = (1000, 1000)
A = np.random.rand(*size)
B = np.random.rand(*size)

# NumPy Multiplication
start = time.time()
C_numpy = np.dot(A, B)
print("NumPy Time:", time.time() - start)

# TensorFlow Multiplication (uses GPU Runtime if available in Colab)
A_tf = tf.constant(A)
B_tf = tf.constant(B)
start = time.time()
C_tf = tf.matmul(A_tf, B_tf)
print("TensorFlow Time:", time.time() - start)


### Example 4: Activation Functions in Keras

In [ ]:
from tensorflow.keras.layers import Dense

# Creating a Dense layer with sigmoid activation
layer = Dense(64, activation="sigmoid")
print("Dense layer created with sigmoid activation:", layer)


### Example 5: Manually Training a Network in NumPy vs Keras

In [ ]:
import numpy as np

# Manually training a network in NumPy
np.random.seed(42)
x_train_demo = np.random.randn(100, 3)
y_train_demo = np.random.randn(100, 1)
weights = np.random.randn(3, 1)
learning_rate = 0.01

for epoch in range(10):
    # Forward pass
    y_pred = np.dot(x_train_demo, weights)
    # Compute loss
    loss = np.mean((y_pred - y_train_demo) ** 2)
    # Compute gradients manually
    gradients = 2 * np.dot(x_train_demo.T, (y_pred - y_train_demo)) / len(x_train_demo)
    # Update weights
    weights -= learning_rate * gradients
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}")


### Example 6: Pre-built Layers in Keras

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model_demo = Sequential(
    [Dense(64, activation="relu", input_shape=(784,)), Dense(10, activation="softmax")]
)
print("Model with pre-built layers created successfully.")
model_demo.summary()


### Example 7: SGD Optimizer — NumPy vs Keras

In [ ]:
import numpy as np

# Implementing SGD in NumPy
learning_rate = 0.01
weights = np.random.randn(3, 3)

for _ in range(100):  # Training loop
    gradient = np.random.randn(3, 3)  # Fake gradient for illustration
    weights -= learning_rate * gradient

print("Final weights after SGD (NumPy):\n", weights)


In [ ]:
from tensorflow.keras.optimizers import SGD

# SGD Optimizer in Keras
optimizer = SGD(learning_rate=0.01)
print("SGD Optimizer created:", optimizer)


---
## Section 3: Understanding Fully Connected Layers
### Example 8: Dense Layer in Keras

In [ ]:
from tensorflow.keras.layers import Dense

# Syntax of Dense Layer
# layer = Dense(units, activation=None, use_bias=True, kernel_initializer="glorot_uniform")

# Example: A Dense Layer with 64 Neurons and sigmoid Activation
layer = Dense(
    64,
    activation="sigmoid",
)  # 64 neurons with sigmoid activation
print("Dense layer with sigmoid:", layer.get_config())


Dense layer with sigmoid: {'name': 'dense', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}, 'units': 64, 'activation': 'sigmoid', 'use_bias': True, 'kernel_initializer': {'module': 'keras.initializers', 'class_name': 'GlorotUniform', 'config': {'seed': None}, 'registered_name': None}, 'bias_initializer': {'module': 'keras.initializers', 'class_name': 'Zeros', 'config': {}, 'registered_name': None}, 'kernel_regularizer': None, 'bias_regularizer': None, 'kernel_constraint': None, 'bias_constraint': None, 'quantization_config': None}


---
## Section 4: Building a Simple Fully Connected Neural Network in Keras
### Step 1: Load and Preprocess Data (Manual Method using PIL)
> **Note:** This method is for when you have your own dataset structured as `dataset/Train/digit_0/`, etc.

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from PIL import Image  # Import Pillow

# Define dataset paths
train_dir = "dataset/Train/"
test_dir = "dataset/Test/"

# Define image size
img_height, img_width = 28, 28


# Function to load images and labels using PIL
def load_images_from_folder(folder):
    images = []
    labels = []
    class_names = sorted(
        os.listdir(folder)
    )  # Sorted class names (digit_0, digit_1, ...)
    class_map = {
        name: i for i, name in enumerate(class_names)
    }  # Map class names to labels

    for class_name in class_names:
        class_path = os.path.join(folder, class_name)
        label = class_map[class_name]
        for filename in os.listdir(class_path):
            img_path = os.path.join(class_path, filename)
            img = Image.open(img_path).convert("L")  # Convert to grayscale
            img = img.resize((img_width, img_height))  # Resize to (28,28)
            img = np.array(img) / 255.0  # Normalize pixel values to [0,1]
            images.append(img)
            labels.append(label)

    return np.array(images), np.array(labels)


# NOTE: Uncomment the lines below only if you have the dataset available.
# x_train, y_train = load_images_from_folder(train_dir)
# x_test,  y_test  = load_images_from_folder(test_dir)

# Reshape images for Keras input
# x_train = x_train.reshape(-1, img_height, img_width, 1)  # (num_samples, 28, 28, 1)
# x_test  = x_test.reshape(-1, img_height, img_width, 1)

# One-hot encode labels
# y_train = to_categorical(y_train, num_classes=10)
# y_test  = to_categorical(y_test,  num_classes=10)

# Print dataset shape
# print(f"Training set: {x_train.shape}, Labels: {y_train.shape}")
# print(f"Testing set:  {x_test.shape},  Labels: {y_test.shape}")

# Visualize some images
# plt.figure(figsize=(10, 4))
# for i in range(10):
#     plt.subplot(2, 5, i + 1)
#     plt.imshow(x_train[i].reshape(28, 28), cmap='gray')
#     plt.title(f"Label: {np.argmax(y_train[i])}")
#     plt.axis("off")
# plt.show()

print(
    "PIL data loading function defined. Uncomment the lines above when dataset is available."
)


### Step 1b: Loading and Preprocessing the MNIST Dataset (from Keras)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the images to values between 0 and 1
x_train, x_test = x_train / 255.0, x_test / 255.0

# Flatten the 28x28 images into 784-dimensional vectors
x_train = x_train.reshape(-1, 28 * 28)
x_test = x_test.reshape(-1, 28 * 28)

# One-hot encode the labels (0–9) for classification
y_train_ohe = tf.keras.utils.to_categorical(y_train, 10)
y_test_ohe = tf.keras.utils.to_categorical(y_test, 10)

print(f"Training set:  {x_train.shape}, Labels: {y_train_ohe.shape}")
print(f"Testing set:   {x_test.shape},  Labels: {y_test_ohe.shape}")


### Visualise Sample MNIST Images

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].reshape(28, 28), cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.suptitle("Sample MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()


---
### Step 2a: Build the Model — Sequential API

In [ ]:
import tensorflow as tf
from tensorflow import keras

num_classes = 10
input_shape = (28, 28, 1)

model = keras.Sequential(
    [
        keras.layers.Input(shape=input_shape),
        keras.layers.Flatten(),  # Flatten 28x28x1 → 784
        keras.layers.Dense(64, activation="sigmoid"),
        keras.layers.Dense(128, activation="sigmoid"),
        keras.layers.Dense(256, activation="sigmoid"),
        keras.layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()


### Step 2b: Build the Model — Functional API

In [ ]:
import tensorflow as tf
from tensorflow import keras

num_classes = 10
input_shape = (28, 28, 1)


def build_functional_model():
    # Input layer
    inputs = keras.Input(shape=input_shape)
    # Flatten layer
    x = keras.layers.Flatten()(inputs)
    # Hidden layers
    x = keras.layers.Dense(64, activation="sigmoid")(x)
    x = keras.layers.Dense(128, activation="sigmoid")(x)
    x = keras.layers.Dense(256, activation="sigmoid")(x)
    # Output layer
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    # Create model
    model = keras.Model(inputs=inputs, outputs=outputs)
    return model


# Build the model
functional_model = build_functional_model()
functional_model.summary()


---
### Step 3: Compiling the Model
We continue with the Sequential model from Step 2a.

In [ ]:
# Compile the Sequential model
model.compile(
    optimizer="sgd",  # Stochastic Gradient Descent
    loss="categorical_crossentropy",  # Loss function for multi-class classification
    metrics=["accuracy"],  # Track accuracy during training
)
print("Model compiled successfully.")


### Step 4: Training the Model with Callbacks

In [ ]:
# Prepare data with the correct shape for the model (28, 28, 1)
(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = (
    tf.keras.datasets.mnist.load_data()
)
x_train_4d = (x_train_raw / 255.0).reshape(-1, 28, 28, 1)
x_test_4d = (x_test_raw / 255.0).reshape(-1, 28, 28, 1)
y_train_cat = tf.keras.utils.to_categorical(y_train_raw, 10)
y_test_cat = tf.keras.utils.to_categorical(y_test_raw, 10)

batch_size = 128
epochs = 20  # Reduced from 2000 for demonstration

# Callbacks
callbacks = [
    keras.callbacks.ModelCheckpoint(filepath="model_at_epoch_{epoch}.keras"),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4),
]

# Train the model with callbacks and validation split
history = model.fit(
    x_train_4d,
    y_train_cat,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.15,
    callbacks=callbacks,
)


### Visualising the Model's Training Progress (Loss & Accuracy)

In [ ]:
import matplotlib.pyplot as plt

# Extracting training and validation loss
train_loss = history.history["loss"]
val_loss = history.history["val_loss"]

# Extracting training and validation accuracy
train_acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]

# Plotting training and validation loss
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(train_loss) + 1), train_loss, label="Training Loss", color="blue")
plt.plot(range(1, len(val_loss) + 1), val_loss, label="Validation Loss", color="orange")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()

# Plotting training and validation accuracy
plt.subplot(1, 2, 2)
plt.plot(
    range(1, len(train_acc) + 1), train_acc, label="Training Accuracy", color="blue"
)
plt.plot(
    range(1, len(val_acc) + 1), val_acc, label="Validation Accuracy", color="orange"
)
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


---
### Step 5: Evaluating the Model

In [ ]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_test_4d, y_test_cat, verbose=2)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")


---
### Step 6: Making Predictions

In [ ]:
import numpy as np

# Predict on test data
predictions = model.predict(x_test_4d)

# Convert predictions from probabilities to digit labels
predicted_labels = np.argmax(predictions, axis=1)

# Check the first prediction
print(f"Predicted label for first image: {predicted_labels[0]}")
print(f"True label for first image:      {np.argmax(y_test_cat[0])}")


In [ ]:
# Visualise predictions on the first 10 test images
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test_4d[i].reshape(28, 28), cmap="gray")
    true_label = np.argmax(y_test_cat[i])
    pred_label = predicted_labels[i]
    colour = "green" if pred_label == true_label else "red"
    plt.title(f"T:{true_label} P:{pred_label}", color=colour)
    plt.axis("off")
plt.suptitle("Predictions (Green=Correct, Red=Wrong)", fontsize=12)
plt.tight_layout()
plt.show()


---
### Step 7: Saving and Loading the Model

In [ ]:
# Save the trained model
model.save("mnist_fully_connected_model.h5")
print("Model saved to 'mnist_fully_connected_model.h5'")


In [ ]:
import tensorflow as tf

# Load the saved model
loaded_model = tf.keras.models.load_model("mnist_fully_connected_model.h5")
print("Model loaded successfully.")

# Re-evaluate the loaded model
loaded_loss, loaded_acc = loaded_model.evaluate(x_test_4d, y_test_cat, verbose=2)
print(f"Loaded Model — Test Loss: {loaded_loss:.4f} | Test Accuracy: {loaded_acc:.4f}")


---
## Summary

| Step | What We Did |
|------|-------------|
| 1 | Installed Keras/TensorFlow and checked versions |
| 2 | Demonstrated why Keras is preferred over raw NumPy |
| 3 | Explored Dense layers and activation functions |
| 4 | Loaded and preprocessed MNIST data |
| 5 | Built FCN models with Sequential and Functional APIs |
| 6 | Compiled with SGD + categorical crossentropy |
| 7 | Trained with `model.fit()`, EarlyStopping, and ModelCheckpoint callbacks |
| 8 | Visualised training loss & accuracy curves |
| 9 | Evaluated with `model.evaluate()` |
| 10 | Made predictions with `model.predict()` |
| 11 | Saved and reloaded the model with `.h5` format |

---
*Good Luck with your Devnagari Digit Classification exercise!*